# Graph neural networks: SM vs BSM (GCN and GAT)

Five-node fully connected graphs with global event features (`data.u`). Event-type stratified undersampling matches the thesis pipeline.


**Note**: PyTorch Geometric is required. Install with `pip install torch-geometric` (see `requirements.txt`).

In [ ]:
import pandas as pd
import numpy as np
import h5py
import matplotlib.pyplot as plt
from plot_style import apply_publication_style
apply_publication_style()
import time
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool, global_max_pool, BatchNorm

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

os.makedirs('figures/exploratory', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('metrics', exist_ok=True)

## 1. Load Data

In [ ]:
def load_dataset(filepath):
    with h5py.File(filepath, 'r') as f:
        df = pd.DataFrame.from_records(f['ForAnalysis/1d'][:])
    return df

sm_df = load_dataset('datasets/new_Input_bbyy_SMEFT_SM_4thMarch_2026.h5')
cbbim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbbim_4thMarch_2026.h5')
cbgim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbgim_4thMarch_2026.h5')
cbhim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbhim_4thMarch_2026.h5')
chbtil_df = load_dataset('datasets/new_Input_bbyy_SMEFT_chbtil_4thMarch_2026.h5')
chgtil_df = load_dataset('datasets/new_Input_bbyy_SMEFT_chgtil_4thMarch_2026.h5')
ctbim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_ctbim_4thMarch_2026.h5')

print(f"SM events: {len(sm_df):,}")
print(f"cbbim events: {len(cbbim_df):,}")
print(f"cbgim events: {len(cbgim_df):,}")
print(f"cbhim events: {len(cbhim_df):,}")
print(f"chbtil events: {len(chbtil_df):,}")
print(f"chgtil events: {len(chgtil_df):,}")
print(f"ctbim events: {len(ctbim_df):,}")

bsm_datasets = {
    'cbbim': cbbim_df,
    'cbgim': cbgim_df,
    'cbhim': cbhim_df,
    'chbtil': chbtil_df,
    'chgtil': chgtil_df,
    'ctbim': ctbim_df,
}

## 2. Convert Events to Graphs

Each event becomes a graph where:
- **Nodes**: Physics objects (Higgs, Lead Jet, SubLead Jet, Lead Photon, SubLead Photon)
- **Node features**: (pT, Eta, Phi, Mass) for each object (4 features per node)
- **Edges**: Fully connected graph (all objects interact)
- **Global features**: 10 event-level observables (m_bbyy, pT_jj, Eta_jj, DPhi_bb, signed_DeltaPhi_jj, cosThetaStar, costheta1, costheta2, DPhi_yybb, Eta_yybb)

In [ ]:
def event_to_graph(row, label):
    """
    Convert a single event to a PyTorch Geometric graph.
    
    Nodes (5 objects):
    - 0: Higgs candidate
    - 1: Lead Jet
    - 2: SubLead Jet  
    - 3: Lead Photon
    - 4: SubLead Photon
    
    Node features: [pT, Eta, Phi, Mass]
    """
    # Node features: [pT, Eta, Phi, Mass] for each object
    node_features = torch.tensor([
        # Higgs
        [row['Higgs_pT'], row['Higgs_Eta'], row['Higgs_Phi'], row['Higgs_Mass']],
        # Lead Jet
        [row['LeadJet_pT'], row['LeadJet_Eta'], row['LeadJet_Phi'], row['LeadJet_M']],
        # SubLead Jet
        [row['SubLeadJet_pT'], row['SubLeadJet_Eta'], row['SubLeadJet_Phi'], row['SubLeadJet_M']],
        # Lead Photon (Mass = 0 for photons)
        [row['LeadPhoton_pT'], row['LeadPhoton_Eta'], row['LeadPhoton_Phi'], 0.0],
        # SubLead Photon
        [row['SubLeadPhoton_pT'], row['SubLeadPhoton_Eta'], row['SubLeadPhoton_Phi'], 0.0],
    ], dtype=torch.float)
    
    # Fully connected edges (all pairs, both directions)
    num_nodes = 5
    edge_index = []
    for i in range(num_nodes):
        for j in range(num_nodes):
            if i != j:
                edge_index.append([i, j])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    
    # Global features (event-level observables)
    global_features = torch.tensor([
        row['m_bbyy'],
        row['pT_jj'],
        row['Eta_jj'],
        row['DPhi_bb'],
        row['signed_DeltaPhi_jj'],
        row['cosThetaStar'],
        row['costheta1'],
        row['costheta2'],
        row['DPhi_yybb'],
        row['Eta_yybb'],
    ], dtype=torch.float)
    
    # Label
    y = torch.tensor([label], dtype=torch.float)
    
    return Data(x=node_features, edge_index=edge_index, u=global_features, y=y)


def _get_event_type(row):
    """Assign event type: ZH, HH, Single H, or Other (priority order)."""
    if row.get('is_ZHEvent', False):
        return 'ZH'
    if row.get('is_HHEvent', False):
        return 'HH'
    if row.get('is_SingleHiggsEvent', False):
        return 'Single H'
    return 'Other'


def create_graph_dataset(sm_df, bsm_df, balance=True):
    """
    Create graph dataset from SM and BSM dataframes.
    When balance=True, stratifies by event type (HH, Single H, Other; ZH when present)
    so SM and BSM have matched event-type compositions. Each stratum is capped at
    min(n_SM, n_BSM); strata with zero events on either side are skipped.
    """
    graphs = []
    
    if balance:
        sm_copy = sm_df.copy()
        bsm_copy = bsm_df.copy()
        sm_copy['_event_type'] = sm_copy.apply(_get_event_type, axis=1)
        bsm_copy['_event_type'] = bsm_copy.apply(_get_event_type, axis=1)
        sm_samples = []
        bsm_samples = []
        for etype in ['ZH', 'HH', 'Single H', 'Other']:
            sm_sub = sm_copy[sm_copy['_event_type'] == etype]
            bsm_sub = bsm_copy[bsm_copy['_event_type'] == etype]
            n_min = min(len(sm_sub), len(bsm_sub))
            if n_min > 0:
                sm_samples.append(sm_sub.sample(n=n_min, random_state=42))
                bsm_samples.append(bsm_sub.sample(n=n_min, random_state=42))
        if sm_samples and bsm_samples:
            sm_sample = pd.concat(sm_samples, ignore_index=True)
            bsm_sample = pd.concat(bsm_samples, ignore_index=True)
        else:
            n_min = min(len(sm_df), len(bsm_df))
            sm_sample = sm_df.sample(n=n_min, random_state=42)
            bsm_sample = bsm_df.sample(n=n_min, random_state=42)
    else:
        sm_sample = sm_df
        bsm_sample = bsm_df
    
    # Convert SM events (label=0)
    for _, row in sm_sample.iterrows():
        graphs.append(event_to_graph(row, label=0))
    
    # Convert BSM events (label=1)
    for _, row in bsm_sample.iterrows():
        graphs.append(event_to_graph(row, label=1))
    
    return graphs

print("Graph construction functions defined.")

In [ ]:
sample_graph = event_to_graph(sm_df.iloc[0], label=0)
print("Sample Graph Structure:")
print(f"  Nodes: {sample_graph.x.shape[0]}")
print(f"  Node features: {sample_graph.x.shape[1]}")
print(f"  Edges: {sample_graph.edge_index.shape[1]}")
print(f"  Global features: {sample_graph.u.shape[0]}")
print(f"  Label: {sample_graph.y.item()}")
print(f"\nNode features (pT, Eta, Phi, Mass):")
print(sample_graph.x)
print(f"\nGlobal features (m_bbyy, pT_jj, Eta_jj, DPhi_bb, signed_DeltaPhi_jj, cosThetaStar, costheta1, costheta2, DPhi_yybb, Eta_yybb):")
print(sample_graph.u)

In [ ]:
def prepare_data(sm_df, bsm_df, batch_size=64):
    """Create graph dataset, split into train/val/test, and return dataloaders."""
    graph_dataset = create_graph_dataset(sm_df, bsm_df)
    np.random.shuffle(graph_dataset)

    n_total = len(graph_dataset)
    n_train = int(0.7 * n_total)
    n_val = int(0.1 * n_total)

    train_dataset = graph_dataset[:n_train]
    val_dataset = graph_dataset[n_train:n_train+n_val]
    test_dataset = graph_dataset[n_train+n_val:]

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, len(graph_dataset)

print("Data preparation function defined.")

## 3. Define GNN Models

In [ ]:
class GCN_Classifier(nn.Module):
    """
    Graph Convolutional Network for event classification.
    Uses GCNConv layers followed by global pooling and MLP.
    """
    def __init__(self, node_features=4, global_features=10, hidden_dim=64, num_layers=3, dropout=0.3):
        super(GCN_Classifier, self).__init__()
        
        self.num_layers = num_layers
        self.n_global = global_features
        
        # Graph convolution layers
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        # First layer
        self.convs.append(GCNConv(node_features, hidden_dim))
        self.bns.append(BatchNorm(hidden_dim))
        
        # Hidden layers
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(BatchNorm(hidden_dim))
        
        # Combine graph embedding with global features
        combined_dim = hidden_dim * 2 + global_features  # mean + max pooling + global
        
        # MLP classifier
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
        self.dropout = dropout
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        global_features = data.u
        
        # Graph convolutions
        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        
        # Global pooling (both mean and max for richer representation)
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        
        # Reshape global features for batched data
        global_features = global_features.view(-1, self.n_global)
        
        # Concatenate graph embedding with global features
        combined = torch.cat([x_mean, x_max, global_features], dim=1)
        
        # Classification
        out = self.classifier(combined)
        return out.squeeze()


class GAT_Classifier(nn.Module):
    """
    Graph Attention Network for event classification.
    Uses attention mechanism to weight node interactions.
    """
    def __init__(self, node_features=4, global_features=10, hidden_dim=64, num_layers=3, 
                 heads=4, dropout=0.3):
        super(GAT_Classifier, self).__init__()
        
        self.num_layers = num_layers
        self.n_global = global_features
        
        # Graph attention layers
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        # First layer
        self.convs.append(GATConv(node_features, hidden_dim, heads=heads, dropout=dropout))
        self.bns.append(BatchNorm(hidden_dim * heads))
        
        # Hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=heads, dropout=dropout))
            self.bns.append(BatchNorm(hidden_dim * heads))
        
        # Final conv layer (single head)
        self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=dropout))
        self.bns.append(BatchNorm(hidden_dim))
        
        # Combine graph embedding with global features
        combined_dim = hidden_dim * 2 + global_features
        
        # MLP classifier
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
        self.dropout = dropout
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        global_features = data.u
        
        # Graph attention layers
        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        
        # Global pooling
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        
        # Reshape global features
        global_features = global_features.view(-1, self.n_global)
        
        # Concatenate
        combined = torch.cat([x_mean, x_max, global_features], dim=1)
        
        # Classification
        out = self.classifier(combined)
        return out.squeeze()


gcn_test = GCN_Classifier(node_features=4, global_features=10, hidden_dim=64, num_layers=3).to(device)
gat_test = GAT_Classifier(node_features=4, global_features=10, hidden_dim=32, num_layers=3, heads=4).to(device)

print("GCN Model:")
print(f"  Parameters: {sum(p.numel() for p in gcn_test.parameters()):,}")
print("\nGAT Model:")
print(f"  Parameters: {sum(p.numel() for p in gat_test.parameters()):,}")

del gcn_test, gat_test

## 4. Training Loop with Metrics

In [ ]:
def train_gnn(model, train_loader, val_loader, epochs=100, lr=0.001, patience=15):
    """
    Train GNN model with early stopping and comprehensive metrics.
    """
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    metrics = {
        'epoch': [],
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'epoch_time': [],
        'total_time': []
    }
    
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    total_start = time.time()
    
    for epoch in range(epochs):
        epoch_start = time.time()
        
        # Training
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            out = model(batch)
            loss = criterion(out, batch.y.float())
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * batch.num_graphs
            preds = (out > 0.5).float()
            train_correct += (preds == batch.y).sum().item()
            train_total += batch.num_graphs
        
        train_loss /= train_total
        train_acc = train_correct / train_total
        
        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                loss = criterion(out, batch.y.float())
                
                val_loss += loss.item() * batch.num_graphs
                preds = (out > 0.5).float()
                val_correct += (preds == batch.y).sum().item()
                val_total += batch.num_graphs
        
        val_loss /= val_total
        val_acc = val_correct / val_total
        
        epoch_time = time.time() - epoch_start
        total_time = time.time() - total_start
        
        # Store metrics
        metrics['epoch'].append(epoch + 1)
        metrics['train_loss'].append(train_loss)
        metrics['train_acc'].append(train_acc)
        metrics['val_loss'].append(val_loss)
        metrics['val_acc'].append(val_acc)
        metrics['epoch_time'].append(epoch_time)
        metrics['total_time'].append(total_time)
        
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f} | "
                  f"Time: {epoch_time:.2f}s")
        
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    # Restore best model
    model.load_state_dict(best_model_state)
    total_time = time.time() - total_start
    
    return metrics, total_time


def bootstrap_auc(y_true, probs, n_bootstrap=1000, confidence=0.95, random_state=42):
    """Bootstrap AUC: resample test set with replacement, compute AUC per sample."""
    np.random.seed(random_state)
    n = len(y_true)
    aucs = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, size=n, replace=True)
        y_b = y_true[idx]
        p_b = probs[idx]
        if len(np.unique(y_b)) < 2:
            continue
        fpr_b, tpr_b, _ = roc_curve(y_b, p_b)
        aucs.append(auc(fpr_b, tpr_b))
    aucs = np.array(aucs)
    alpha = (1 - confidence) / 2
    ci_lower = np.percentile(aucs, 100 * alpha)
    ci_upper = np.percentile(aucs, 100 * (1 - alpha))
    return np.mean(aucs), np.std(aucs), ci_lower, ci_upper


def evaluate_gnn(model, test_loader, model_name):
    """Evaluate GNN model on test set with comprehensive metrics."""
    model.eval()
    all_probs, all_labels = [], []

    start_time = time.time()
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            probs = model(batch).cpu().numpy()
            labels = batch.y.cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels)

    inference_time = time.time() - start_time
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds = (all_probs > 0.5).astype(int)

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    auc_score = auc(fpr, tpr)
    auc_mean, auc_std, auc_ci_lower, auc_ci_upper = bootstrap_auc(all_labels, all_probs, n_bootstrap=1000)

    print(f"\n{model_name} Test Results:")
    print("="*50)
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  AUC:       {auc_mean:.4f} ± {auc_std:.4f}  (95% CI: [{auc_ci_lower:.4f}, {auc_ci_upper:.4f}])")
    print(f"  Inference: {inference_time:.4f}s ({len(all_labels)/inference_time:,.0f} events/s)")

    return {
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
        'auc': auc_score, 'auc_mean': auc_mean, 'auc_std': auc_std,
        'auc_ci_lower': auc_ci_lower, 'auc_ci_upper': auc_ci_upper,
        'inference_time': inference_time,
        'events_per_second': len(all_labels) / inference_time,
        'probs': all_probs, 'labels': all_labels, 'fpr': fpr, 'tpr': tpr,
    }

In [ ]:
all_results = {}

for bsm_name, bsm_df in bsm_datasets.items():
    print(f"\n{'='*70}")
    print(f"  SM vs {bsm_name}")
    print(f"{'='*70}")

    print(f"Creating graph dataset (SM vs {bsm_name})...")
    start_time = time.time()
    train_loader, val_loader, test_loader, n_graphs = prepare_data(sm_df, bsm_df, batch_size=64)
    conversion_time = time.time() - start_time
    print(f"Created {n_graphs:,} graphs in {conversion_time:.2f}s")

    # --- GCN ---
    gcn_model = GCN_Classifier(node_features=4, global_features=10, hidden_dim=64, num_layers=3).to(device)
    print(f"\nTraining GCN (SM vs {bsm_name})...")
    gcn_metrics, gcn_train_time = train_gnn(gcn_model, train_loader, val_loader, epochs=100, patience=15)
    gcn_eval = evaluate_gnn(gcn_model, test_loader, f"GCN (SM vs {bsm_name})")

    # --- GAT ---
    gat_model = GAT_Classifier(node_features=4, global_features=10, hidden_dim=32, num_layers=3, heads=4).to(device)
    print(f"\nTraining GAT (SM vs {bsm_name})...")
    gat_metrics, gat_train_time = train_gnn(gat_model, train_loader, val_loader, epochs=100, patience=15)
    gat_eval = evaluate_gnn(gat_model, test_loader, f"GAT (SM vs {bsm_name})")

    all_results[bsm_name] = {
        'gcn_model': gcn_model,
        'gat_model': gat_model,
        'gcn_metrics': gcn_metrics,
        'gat_metrics': gat_metrics,
        'gcn_eval': gcn_eval,
        'gat_eval': gat_eval,
        'gcn_train_time': gcn_train_time,
        'gat_train_time': gat_train_time,
        'test_loader': test_loader,
    }

    # Save models per BSM operator
    torch.save({
        'model_state_dict': gcn_model.state_dict(),
        'model_type': 'GCN',
        'bsm_operator': bsm_name,
        'auc': gcn_eval['auc'],
        'accuracy': gcn_eval['accuracy'],
    }, f'models/gcn_{bsm_name}_classifier.pt')

    torch.save({
        'model_state_dict': gat_model.state_dict(),
        'model_type': 'GAT',
        'bsm_operator': bsm_name,
        'auc': gat_eval['auc'],
        'accuracy': gat_eval['accuracy'],
    }, f'models/gat_{bsm_name}_classifier.pt')

    pd.DataFrame(gcn_metrics).to_csv(f'metrics/gcn_{bsm_name}_training_metrics.csv', index=False)
    pd.DataFrame(gat_metrics).to_csv(f'metrics/gat_{bsm_name}_training_metrics.csv', index=False)

print(f"\n{'='*70}")
print("All training complete!")
print(f"{'='*70}")

In [ ]:
# Quick overview of results so far
for bsm_name, res in all_results.items():
    gcn_auc = res['gcn_eval']['auc']
    gat_auc = res['gat_eval']['auc']
    print(f"SM vs {bsm_name:8s}  |  GCN AUC: {gcn_auc:.4f}  |  GAT AUC: {gat_auc:.4f}")

## 5. Training Curves

In [ ]:
n_bsm = len(all_results)
fig, axes = plt.subplots(n_bsm, 4, figsize=(22, 4 * n_bsm))
if n_bsm == 1:
    axes = axes[np.newaxis, :]

for row, (bsm_name, res) in enumerate(all_results.items()):
    gcn_m = res['gcn_metrics']
    gat_m = res['gat_metrics']

    # GCN Loss
    axes[row, 0].plot(gcn_m['epoch'], gcn_m['train_loss'], label='Train', lw=2)
    axes[row, 0].plot(gcn_m['epoch'], gcn_m['val_loss'], label='Val', lw=2)
    axes[row, 0].set_title(f'{bsm_name} - GCN Loss', fontsize=12)
    axes[row, 0].set_xlabel('Epoch'); axes[row, 0].set_ylabel('Loss')
    axes[row, 0].legend(fontsize=9); axes[row, 0].grid(True, alpha=0.3)

    # GCN Accuracy
    axes[row, 1].plot(gcn_m['epoch'], gcn_m['train_acc'], label='Train', lw=2)
    axes[row, 1].plot(gcn_m['epoch'], gcn_m['val_acc'], label='Val', lw=2)
    axes[row, 1].set_title(f'{bsm_name} - GCN Accuracy', fontsize=12)
    axes[row, 1].set_xlabel('Epoch'); axes[row, 1].set_ylabel('Accuracy')
    axes[row, 1].legend(fontsize=9); axes[row, 1].grid(True, alpha=0.3)

    # GAT Loss
    axes[row, 2].plot(gat_m['epoch'], gat_m['train_loss'], label='Train', lw=2)
    axes[row, 2].plot(gat_m['epoch'], gat_m['val_loss'], label='Val', lw=2)
    axes[row, 2].set_title(f'{bsm_name} - GAT Loss', fontsize=12)
    axes[row, 2].set_xlabel('Epoch'); axes[row, 2].set_ylabel('Loss')
    axes[row, 2].legend(fontsize=9); axes[row, 2].grid(True, alpha=0.3)

    # GAT Accuracy
    axes[row, 3].plot(gat_m['epoch'], gat_m['train_acc'], label='Train', lw=2)
    axes[row, 3].plot(gat_m['epoch'], gat_m['val_acc'], label='Val', lw=2)
    axes[row, 3].set_title(f'{bsm_name} - GAT Accuracy', fontsize=12)
    axes[row, 3].set_xlabel('Epoch'); axes[row, 3].set_ylabel('Accuracy')
    axes[row, 3].legend(fontsize=9); axes[row, 3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/exploratory/gnn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Evaluation on Test Set

In [ ]:
print("Evaluation results per BSM operator:")
for bsm_name, res in all_results.items():
    gcn_e = res['gcn_eval']
    gat_e = res['gat_eval']
    print(f"\n  {bsm_name}:")
    print(f"    GCN - AUC: {gcn_e['auc']:.4f}, Acc: {gcn_e['accuracy']:.4f}, F1: {gcn_e['f1']:.4f}")
    print(f"    GAT - AUC: {gat_e['auc']:.4f}, Acc: {gat_e['accuracy']:.4f}, F1: {gat_e['f1']:.4f}")

## 6b. Feature importance (permutation on global features)

Permutation importance on the 10 global features — compare with EDA overlap/ratio findings.

In [ ]:
GLOBAL_FEATURE_NAMES = ['m_bbyy', 'pT_jj', 'Eta_jj', 'DPhi_bb', 'signed_DeltaPhi_jj', 
                       'cosThetaStar', 'costheta1', 'costheta2', 'DPhi_yybb', 'Eta_yybb']

def gnn_permutation_importance(model, test_loader, feature_idx, device):
    """Permute global feature at feature_idx and return accuracy."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            u = batch.u.clone()
            # Batched u is flattened (n_graphs*10,); reshape to (n_graphs, 10)
            n_global = 10
            u = u.view(-1, n_global)
            perm_idx = torch.randperm(u.size(0), device=u.device)
            u[:, feature_idx] = u[perm_idx, feature_idx]
            batch_perm = batch.clone()
            batch_perm.u = u
            out = model(batch_perm)
            preds = (out > 0.5).float()
            correct += (preds == batch.y).sum().item()
            total += batch.num_graphs
    return correct / total if total > 0 else 0

# Compute permutation importance for GCN and GAT per operator
gnn_importances = {}  # {bsm_name: {'GCN': {feat: imp}, 'GAT': {feat: imp}}}
for bsm_name, res in all_results.items():
    test_loader = res['test_loader']
    baseline_gcn = res['gcn_eval']['accuracy']
    baseline_gat = res['gat_eval']['accuracy']
    imp_gcn, imp_gat = {}, {}
    for i, feat in enumerate(GLOBAL_FEATURE_NAMES):
        acc_perm_gcn = gnn_permutation_importance(res['gcn_model'], test_loader, i, device)
        acc_perm_gat = gnn_permutation_importance(res['gat_model'], test_loader, i, device)
        imp_gcn[feat] = baseline_gcn - acc_perm_gcn
        imp_gat[feat] = baseline_gat - acc_perm_gat
    gnn_importances[bsm_name] = {'GCN': imp_gcn, 'GAT': imp_gat}

# Plot top features per operator (GCN)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for idx, (bsm_name, imps) in enumerate(gnn_importances.items()):
    ax = axes[idx]
    feat_imp = sorted(imps['GCN'].items(), key=lambda x: -x[1])
    feats = [f[0] for f in feat_imp]
    vals = [f[1] for f in feat_imp]
    ax.barh(feats, vals, color='steelblue', alpha=0.8)
    ax.set_xlabel('Accuracy drop (importance)')
    ax.set_title(f'{bsm_name} - GCN')
    ax.invert_yaxis()
plt.suptitle('GNN feature importance (GCN, global features)', fontsize=12)
plt.tight_layout()
plt.savefig('figures/exploratory/gnn_feature_importance_gcn.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# GAT feature importance plot
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for idx, (bsm_name, imps) in enumerate(gnn_importances.items()):
    ax = axes[idx]
    feat_imp = sorted(imps['GAT'].items(), key=lambda x: -x[1])
    feats = [f[0] for f in feat_imp]
    vals = [f[1] for f in feat_imp]
    ax.barh(feats, vals, color='darkorange', alpha=0.8)
    ax.set_xlabel('Accuracy drop (importance)')
    ax.set_title(f'{bsm_name} - GAT')
    ax.invert_yaxis()
plt.suptitle('GNN feature importance (GAT, global features)', fontsize=12)
plt.tight_layout()
plt.savefig('figures/exploratory/gnn_feature_importance_gat.png', dpi=150, bbox_inches='tight')
plt.show()

# Export raw importance scores
rows = []
for bsm_name, imps in gnn_importances.items():
    for arch in ['GCN', 'GAT']:
        for feat, val in imps[arch].items():
            rows.append({'operator': bsm_name, 'model': arch, 'feature': feat, 'importance': val})
imp_df = pd.DataFrame(rows)
imp_df.to_csv('metrics/gnn_feature_importance.csv', index=False)
print(f"Exported to metrics/gnn_feature_importance.csv ({len(rows)} rows)")
print("\nTop 5 per operator (GCN):")
for op in imp_df['operator'].unique():
    top = imp_df[(imp_df['operator']==op) & (imp_df['model']=='GCN')].nlargest(5, 'importance')
    print(f"  {op}: {list(zip(top['feature'], top['importance'].round(4)))}")

## 6c. Learned output distribution (P(BSM) for SM vs BSM)

Histograms of predicted P(BSM) on test set — GCN and GAT per operator.

In [ ]:
# Learned output distribution: GCN
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
for idx, (bsm_name, res) in enumerate(all_results.items()):
    ax = axes[idx]
    p_bsm = np.asarray(res['gcn_eval']['probs']).flatten()
    labels = np.asarray(res['gcn_eval']['labels']).flatten()
    sm_mask = labels == 0
    bsm_mask = labels == 1
    ax.hist(p_bsm[sm_mask], bins=30, label='SM', density=True, color='C0', histtype='step', linewidth=2)
    ax.hist(p_bsm[bsm_mask], bins=30, label='BSM', density=True, color='C1', histtype='step', linewidth=2)
    ax.set_xlabel('P(BSM)')
    ax.set_ylabel('Density')
    ax.set_title(f'{bsm_name} - GCN')
    ax.legend()
    ax.set_xlim(0, 1)
plt.suptitle('GCN learned output distribution (test set)', fontsize=12)
plt.tight_layout()
plt.savefig('figures/exploratory/gnn_learned_distribution_gcn.png', dpi=150, bbox_inches='tight')
plt.show()

# Learned output distribution: GAT
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
for idx, (bsm_name, res) in enumerate(all_results.items()):
    ax = axes[idx]
    p_bsm = np.asarray(res['gat_eval']['probs']).flatten()
    labels = np.asarray(res['gat_eval']['labels']).flatten()
    sm_mask = labels == 0
    bsm_mask = labels == 1
    ax.hist(p_bsm[sm_mask], bins=30, label='SM', density=True, color='C0', histtype='step', linewidth=2)
    ax.hist(p_bsm[bsm_mask], bins=30, label='BSM', density=True, color='C1', histtype='step', linewidth=2)
    ax.set_xlabel('P(BSM)')
    ax.set_ylabel('Density')
    ax.set_title(f'{bsm_name} - GAT')
    ax.legend()
    ax.set_xlim(0, 1)
plt.suptitle('GAT learned output distribution (test set)', fontsize=12)
plt.tight_layout()
plt.savefig('figures/exploratory/gnn_learned_distribution_gat.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
n_bsm = len(all_results)
fig, axes = plt.subplots(n_bsm, 2, figsize=(14, 5 * n_bsm))
if n_bsm == 1:
    axes = axes[np.newaxis, :]

colors_gcn = plt.cm.tab10(np.linspace(0, 1, 10))
colors_gat = plt.cm.Set2(np.linspace(0, 1, 8))

for row, (bsm_name, res) in enumerate(all_results.items()):
    gcn_e = res['gcn_eval']
    gat_e = res['gat_eval']

    # ROC curves
    axes[row, 0].plot(gcn_e['fpr'], gcn_e['tpr'], color='blue', lw=2,
                      label=f"GCN (AUC={gcn_e['auc']:.4f})")
    axes[row, 0].plot(gat_e['fpr'], gat_e['tpr'], color='red', lw=2,
                      label=f"GAT (AUC={gat_e['auc']:.4f})")
    axes[row, 0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    axes[row, 0].set_xlabel('False Positive Rate')
    axes[row, 0].set_ylabel('True Positive Rate')
    axes[row, 0].set_title(f'ROC: SM vs {bsm_name}', fontsize=13)
    axes[row, 0].legend(fontsize=10); axes[row, 0].grid(True, alpha=0.3)

    # Score distributions (GCN)
    axes[row, 1].hist(gcn_e['probs'][gcn_e['labels']==0], bins=50, density=True,
                      histtype='step', linewidth=2, label='SM (GCN)', color='blue')
    axes[row, 1].hist(gcn_e['probs'][gcn_e['labels']==1], bins=50, density=True,
                      histtype='step', linewidth=2, label=f'{bsm_name} (GCN)', color='red')
    axes[row, 1].set_xlabel('Output Score')
    axes[row, 1].set_ylabel('Density')
    axes[row, 1].set_title(f'GCN Score Distribution: SM vs {bsm_name}', fontsize=13)
    axes[row, 1].legend(fontsize=10); axes[row, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/exploratory/gnn_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
n_bsm = len(all_results)
fig, axes = plt.subplots(n_bsm, 2, figsize=(12, 5 * n_bsm))
if n_bsm == 1:
    axes = axes[np.newaxis, :]

for row, (bsm_name, res) in enumerate(all_results.items()):
    for col, (eval_results, arch) in enumerate([(res['gcn_eval'], 'GCN'), (res['gat_eval'], 'GAT')]):
        ax = axes[row, col]
        preds = (eval_results['probs'] > 0.5).astype(int)
        cm = confusion_matrix(eval_results['labels'], preds)
        cm_frac = cm.astype('float') / cm.sum(axis=1, keepdims=True)

        im = ax.imshow(cm_frac, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
        ax.figure.colorbar(im, ax=ax, format='%.2f')
        ax.set(xticks=[0, 1], yticks=[0, 1],
               xticklabels=['SM', bsm_name], yticklabels=['SM', bsm_name],
               xlabel='Predicted', ylabel='True',
               title=f'{arch}: SM vs {bsm_name}')

        for i in range(2):
            for j in range(2):
                ax.text(j, i, f'{cm_frac[i, j]:.3f}\n({cm[i, j]:,})',
                        ha="center", va="center",
                        color="white" if cm_frac[i, j] > 0.5 else "black",
                        fontsize=12)

plt.tight_layout()
plt.savefig('figures/exploratory/gnn_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Models and Metrics

In [ ]:
rows = []
for bsm_name, res in all_results.items():
    for arch, key_prefix in [('GCN', 'gcn'), ('GAT', 'gat')]:
        ev = res[f'{key_prefix}_eval']
        rows.append({
            'BSM_Operator': bsm_name,
            'Architecture': arch,
            'Accuracy': ev['accuracy'],
            'Precision': ev['precision'],
            'Recall': ev['recall'],
            'F1': ev['f1'],
            'AUC': ev['auc'],
            'AUC_mean': ev['auc_mean'],
            'AUC_std': ev['auc_std'],
            'AUC_ci_lower': ev['auc_ci_lower'],
            'AUC_ci_upper': ev['auc_ci_upper'],
            'Training_Time_s': res[f'{key_prefix}_train_time'],
            'Inference_Time_s': ev['inference_time'],
            'Events_per_Second': ev['events_per_second'],
        })

comparison_df = pd.DataFrame(rows)
comparison_df.to_csv('metrics/gnn_comparison.csv', index=False)
print("Comparison metrics saved to metrics/gnn_comparison.csv")
print(comparison_df.to_string(index=False))

## 8. Summary

In [ ]:
print("="*70)
print("GNN CLASSIFICATION SUMMARY - ALL BSM OPERATORS")
print("="*70)

print("\nGraph Structure:")
print(f"   Nodes per event: 5 (Higgs, LeadJet, SubLeadJet, LeadPhoton, SubLeadPhoton)")
print(f"   Node features: 4 (pT, Eta, Phi, Mass)")
print(f"   Edges: 20 (fully connected)")
print(f"   Global features: 10 (m_bbyy, pT_jj, Eta_jj, DPhi_bb, signed_DeltaPhi_jj, cosThetaStar, costheta1, costheta2, DPhi_yybb, Eta_yybb)")

for bsm_name, res in all_results.items():
    gcn_e = res['gcn_eval']
    gat_e = res['gat_eval']
    print(f"\n--- SM vs {bsm_name} ---")
    print(f"  GCN  AUC: {gcn_e['auc']:.4f}  Acc: {gcn_e['accuracy']:.4f}  F1: {gcn_e['f1']:.4f}  Train: {res['gcn_train_time']:.1f}s")
    print(f"  GAT  AUC: {gat_e['auc']:.4f}  Acc: {gat_e['accuracy']:.4f}  F1: {gat_e['f1']:.4f}  Train: {res['gat_train_time']:.1f}s")

print("\n" + "="*70)
print("\nFull Comparison Table:")
print(comparison_df.to_string(index=False))

In [ ]:
bsm_names = list(all_results.keys())
gcn_aucs = [all_results[b]['gcn_eval']['auc_mean'] for b in bsm_names]
gat_aucs = [all_results[b]['gat_eval']['auc_mean'] for b in bsm_names]
gcn_auc_errs = [all_results[b]['gcn_eval']['auc_std'] for b in bsm_names]
gat_auc_errs = [all_results[b]['gat_eval']['auc_std'] for b in bsm_names]
gcn_accs = [all_results[b]['gcn_eval']['accuracy'] for b in bsm_names]
gat_accs = [all_results[b]['gat_eval']['accuracy'] for b in bsm_names]

x = np.arange(len(bsm_names))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(x - width/2, gcn_aucs, width, yerr=gcn_auc_errs, capsize=3, label='GCN', color='steelblue', alpha=0.8)
axes[0].bar(x + width/2, gat_aucs, width, yerr=gat_auc_errs, capsize=3, label='GAT', color='coral', alpha=0.8)
axes[0].set_ylabel('AUC', fontsize=13)
axes[0].set_title('AUC by BSM Operator', fontsize=14)
axes[0].set_xticks(x)
axes[0].set_xticklabels(bsm_names, rotation=30, ha='right')
axes[0].legend(fontsize=11)
axes[0].set_ylim(0.4, 1.0)
axes[0].grid(True, alpha=0.3, axis='y')
for i, (g, a) in enumerate(zip(gcn_aucs, gat_aucs)):
    axes[0].text(i - width/2, g + 0.01, f'{g:.3f}', ha='center', va='bottom', fontsize=8)
    axes[0].text(i + width/2, a + 0.01, f'{a:.3f}', ha='center', va='bottom', fontsize=8)

axes[1].bar(x - width/2, gcn_accs, width, label='GCN', color='steelblue', alpha=0.8)
axes[1].bar(x + width/2, gat_accs, width, label='GAT', color='coral', alpha=0.8)
axes[1].set_ylabel('Accuracy', fontsize=13)
axes[1].set_title('Accuracy by BSM Operator', fontsize=14)
axes[1].set_xticks(x)
axes[1].set_xticklabels(bsm_names, rotation=30, ha='right')
axes[1].legend(fontsize=11)
axes[1].set_ylim(0.4, 1.0)
axes[1].grid(True, alpha=0.3, axis='y')
for i, (g, a) in enumerate(zip(gcn_accs, gat_accs)):
    axes[1].text(i - width/2, g + 0.01, f'{g:.3f}', ha='center', va='bottom', fontsize=8)
    axes[1].text(i + width/2, a + 0.01, f'{a:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('figures/exploratory/gnn_bsm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()